# Mini-TP 4 — Puntúa tu modelo sobre un flujo (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Aplica **tu modelo** (el de la Sesión 1) sobre un **flujo de eventos** en lugar de pedidos sueltos. Completa las celdas marcadas con `# TODO`.

**Qué debes entregar**
1. Un **flujo** de eventos con las *features* de tu modelo (simulado, o consumido de Kafka).
2. Un **consumidor** que puntúe tu modelo sobre **cada** evento (inferencia online).
3. **Métricas por ventana:** throughput (ev/s), latencia **p95** y un indicador simple de **drift** de entradas.
4. Una **alerta** cuando una métrica cruce un umbral.
5. Una **comparación** contra puntuar el mismo lote en **batch**, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; inferencia online sobre el flujo; métricas por ventana correctas; y la reflexión streaming vs batch.

> Apóyate en `streaming_tutorial.ipynb`. Entorno uv: `uv add scikit-learn joblib numpy kafka-python`.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "numpy"])
print("Listo.")

## 1. Tu modelo

Carga **tu** modelo. Si aún no lo tienes a mano, el placeholder deja probar la mecánica.

In [2]:
# TODO: carga tu modelo (debe exponer .predict(X))
import os,mlflow,numpy as np
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin123"

mlflow.set_tracking_uri("http://localhost:5000")
modelo = mlflow.xgboost.load_model("models:/xgb_best/2")   # o "models:/xgb_best/latest"
print("Modelo:", type(modelo).__name__)

# Si fuera necesario serializar a .pkl ejecutar pickle.dump
#import pickle
#with open("xgb_best.pkl", "wb") as f:
#    pickle.dump(model, f)

N_FEATURES = modelo.n_features_in_
print("Modelo cargado. n_features =", N_FEATURES)
print("Prediccin de prueba:", modelo.predict(np.zeros((1, N_FEATURES))))

# --- placeholder (reemplázalo por tu modelo) ---
#from sklearn.linear_model import LogisticRegression
#_X = np.random.RandomState(0).randn(300, 3); _y = (_X.sum(1) > 0).astype(int)
#modelo = LogisticRegression().fit(_X, _y)
#N_FEATURES = 3   # TODO: ajusta al número de features de tu modelo
#print("Modelo cargado. n_features =", N_FEATURES)

Modelo: XGBRegressor
Modelo cargado. n_features = 23
Prediccin de prueba: [12.191017]


## 2. El flujo

Simula un flujo con las features de tu modelo. Introduce un cambio de distribución a mitad de camino para poder detectar drift. (Alternativa: consumir de Kafka como en el tutorial.)

In [5]:
import queue, threading, time, random
stream = queue.Queue(); STOP = object()
N = 8000

def productor():
    for i in range(N):
        # TODO: genera un vector de N_FEATURES acorde a tu dominio
        mu = 0.0 if i < N//2 else 1.2
        stream.put({"values": [random.gauss(mu, 1) for _ in range(N_FEATURES)], "t": time.time()})
        time.sleep(0.0005)
    stream.put(STOP)
print("Productor listo.")

Productor listo.


## 3. Consumidor con inferencia online + métricas por ventana

In [9]:
from collections import deque
import numpy as np

latencias = []; ventana = deque(); alertas = []

def consumidor():
    procesados = 0
    while True:
        ev = stream.get()
        if ev is STOP: break
        t0 = time.perf_counter()
        pred = modelo.predict([ev["values"]])[0]           # inferencia online
        latencias.append((time.perf_counter() - t0) * 1000)

        ventana.append((ev["t"], ev["values"][0], pred))
        ahora = ev["t"]
        while ventana and ahora - ventana[0][0] > 1.0:      # ventana deslizante 1s
            ventana.popleft()

        procesados += 1
        if procesados % 200 == 0:
            media_f0 = float(np.mean([w[1] for w in ventana]))
            # TODO: define tu indicador de drift y tu umbral de alerta
            if media_f0 >1:
                alertas.append((procesados, media_f0))

t0 = time.time()
pt = threading.Thread(target=productor); ct = threading.Thread(target=consumidor)
pt.start(); ct.start(); pt.join(); ct.join()
dur = time.time() - t0

print(f"throughput : {len(latencias)/dur:.0f} ev/s")
print(f"p95        : {np.percentile(latencias, 95):.3f} ms")
print(f"alertas de drift: {alertas}")

throughput : 967 ev/s
p95        : 0.603 ms
alertas de drift: [(4800, 1.0254581098981947), (5000, 1.2150065106295953), (5200, 1.205920148049874), (5400, 1.1729100614673953), (5600, 1.1728285617206142), (5800, 1.1988095294271812), (6000, 1.2069662072562337), (6200, 1.1921251236748613), (6400, 1.1878013745623095), (6600, 1.1897661417279748), (6800, 1.1647595076996637), (7000, 1.140659237622057), (7200, 1.1446417910496625), (7400, 1.1662761569489415), (7600, 1.1547782160816649), (7800, 1.1737400495073944), (8000, 1.183642596383506)]


## 4. Comparación contra batch

Puntúa el mismo conjunto de eventos de una sola vez (batch) y compara tiempo/uso frente al streaming.

In [32]:
# TODO: arma un lote con los mismos vectores y puntúalo de una vez con modelo.predict(LOTE)
from collections import deque
import numpy as np
mu = 0
N = 8000
LOTE = []
for _ in range(N):
    LOTE.append([random.gauss(mu, 1) for _ in range(N_FEATURES)])

# Predicciones en batch
t0_start = time.time()
batch_pred = modelo.predict(LOTE)
dur = time.time() - t0_start
for i in range(N):
    #print("Prediccion del lote item "+str(i)+": "+format(batch_pred[i]))
    pass

print(f"Haciendo prediccion en batch, throughput : {N/dur:.0f} ev/s")

# Esto es si lo quiero hacer de a una predicción, para medir latencias individuales
latencias = []; ventana = deque(); alertas = []
t0_start = time.time()
for i in range(N):
    t0 = time.perf_counter()
    batch_pred = modelo.predict([LOTE[i]])[0]
    latencias.append((time.perf_counter() - t0) * 1000)
    #print("Prediccion: "+format(batch_pred))

dur = time.time() - t0_start
print(f"Haciendo batch pero de a una prediccion por vez, throughput : {len(latencias)/dur:.0f} ev/s")
print(f"Haciendo batch pero de a una prediccion por vez, p95        : {np.percentile(latencias, 95):.3f} ms")

# Compara: ¿latencia por evento? ¿frescura del resultado? ¿cuándo conviene cada uno?
#print("Pendiente: implementar el scoring batch y comparar.")


Haciendo prediccion en batch, throughput : 110479 ev/s
Haciendo batch pero de a una prediccion por vez, throughput : 1150 ev/s
Haciendo batch pero de a una prediccion por vez, p95        : 1.309 ms


### Reflexión (completa)

_TODO: 3–5 líneas. ¿Qué te dio el streaming que el batch no? ¿Cuándo usarías cada uno en tu plataforma? ¿Cómo detectaste el drift y qué harías al dispararse la alerta (reentrenar, avisar, degradar)?_

El streaming me dio resultados al instante, evento por evento (967 ev/s, p95 de 0.6 ms), algo que el batch no ofrece: en batch hay que esperar a juntar todo el lote antes de tener la primera predicción. A cambio, el batch vectorizado (`modelo.predict(LOTE)` de una sola vez) fue muchísimo más rápido en throughput total: 110.479 ev/s, ~114x más que el streaming. Repitiendo el batch pero prediciendo de a un evento por vez (sin vectorizar, sin cola ni threads) el throughput bajó a 1150 ev/s, parecido al streaming — o sea que la gran ganancia de velocidad no viene de "ser batch", sino de vectorizar muchas predicciones juntas en una sola llamada al modelo.

En mi plataforma usaría streaming cuando necesito responder rápido a cada evento en cuanto llega (por ejemplo, decidir algo en el momento). Usaría batch cuando puedo esperar y procesar todo junto (por ejemplo, un job nocturno), porque ahí aprovecho la vectorización y gano mucho throughput.

El drift lo detecté siguiendo el promedio de la feature 0 en una ventana deslizante de 1 segundo, y disparando una alerta cuando ese promedio superaba 1. Como el flujo cambia su distribución a mitad de camino (media 0 → media 1.2), la alerta empezó a dispararse cerca del evento 4800, poco después del cambio real (evento 4000), con el delay esperable de la ventana. Si esto pasara en producción, primero avisaría (alerta a los responsables del modelo) y revisaría si el cambio es real o un error de datos; si se confirma que la distribución de entrada cambió en serio, reentrenaría el modelo con datos más recientes, y mientras tanto podría degradar (usar una regla simple o un modelo más conservador) para no confiar ciegamente en predicciones fuera de la distribución de entrenamiento.


## 5. (Opcional) Con Kafka en Docker

Levanta un broker y reemplaza la cola por un topic real:
```bash
docker run -d --name redpanda -p 9092:9092 \
  redpandadata/redpanda redpanda start --overprovisioned --smp 1 --check=false
uv add kafka-python
```
Produce a un topic `eventos` y consume puntuando tu modelo (ver la parte 6 del tutorial).

In [ ]:
import socket

def hay_broker(host="localhost", port=9092):
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

if not hay_broker():
    print("No hay broker en localhost:9092 -> se omite la parte de Kafka.")
    print("Levanta Redpanda con el comando docker de arriba y vuelve a correr esta celda.")
else:
    from kafka import KafkaProducer, KafkaConsumer
    import json, threading, time, random
    from collections import deque
    import numpy as np

    TOPIC = "eventos"
    N_KAFKA = 2000  # eventos a producir para esta prueba

    # --- Productor: mismo esquema de evento y mismo drift a mitad de camino que en el punto 2 ---
    def productor_kafka():
        prod = KafkaProducer(bootstrap_servers="localhost:9092",
                              value_serializer=lambda v: json.dumps(v).encode())
        for i in range(N_KAFKA):
            mu = 0.0 if i < N_KAFKA // 2 else 1.2
            evento = {"values": [random.gauss(mu, 1) for _ in range(N_FEATURES)], "t": time.time()}
            prod.send(TOPIC, evento)
            time.sleep(0.0005)
        prod.flush()
        prod.close()

    # --- Consumidor: mismo scoring online + métricas por ventana que en el punto 3 ---
    latencias_kafka = []; ventana_kafka = deque(); alertas_kafka = []

    def consumidor_kafka():
        cons = KafkaConsumer(TOPIC, bootstrap_servers="localhost:9092",
                              auto_offset_reset="latest",
                              value_deserializer=lambda b: json.loads(b.decode()),
                              consumer_timeout_ms=8000)  # corta si no llegan más mensajes
        procesados = 0
        for msg in cons:
            ev = msg.value
            t0 = time.perf_counter()
            pred = modelo.predict([ev["values"]])[0]          # mismo scoring de siempre
            latencias_kafka.append((time.perf_counter() - t0) * 1000)

            ventana_kafka.append((ev["t"], ev["values"][0], pred))
            ahora = ev["t"]
            while ventana_kafka and ahora - ventana_kafka[0][0] > 1.0:
                ventana_kafka.popleft()

            procesados += 1
            if procesados % 200 == 0:
                media_f0 = float(np.mean([w[1] for w in ventana_kafka]))
                if media_f0 > 1:
                    alertas_kafka.append((procesados, media_f0))
            if procesados >= N_KAFKA:
                break
        cons.close()

    # El consumidor arranca primero (auto_offset_reset="latest": solo ve lo que se produce desde ahora)
    t0 = time.time()
    ct = threading.Thread(target=consumidor_kafka); ct.start()
    time.sleep(1.0)  # da tiempo a que el consumer group se registre antes de producir
    pt = threading.Thread(target=productor_kafka); pt.start()
    pt.join(); ct.join()
    dur = time.time() - t0

    print(f"Eventos puntuados vía Kafka: {len(latencias_kafka)}")
    if latencias_kafka:
        print(f"throughput : {len(latencias_kafka)/dur:.0f} ev/s")
        print(f"p95        : {np.percentile(latencias_kafka, 95):.3f} ms")
        print(f"alertas de drift: {alertas_kafka[:5]}{' ...' if len(alertas_kafka) > 5 else ''}")
    print("Mismo código de scoring y de métricas que en el punto 3; solo cambió de dónde llegan los eventos.")


C:\Users\rodol\AppData\Local\Temp\ipykernel_18048\2111827884.py:38: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  cons = KafkaConsumer(TOPIC, bootstrap_servers="localhost:9092",


Eventos puntuados vía Kafka: 2000
throughput : 229 ev/s
p95        : 7.091 ms
alertas de drift: [(1400, 1.1327897538535672), (1600, 1.238021954791365), (1800, 1.2082824958407423), (2000, 1.1223336204416505)]
Mismo código de scoring y de métricas que en el punto 3; solo cambió de dónde llegan los eventos.
